# Finger Counter with a CNN

We're going to train a CNN to count the number of fingers (1-5) held up in a photo.

First, submit your data here: https://forms.gle/7ycWoRytFp3WFUY87

Two datasets are available. Change `DATA_FOLDER` in the next cell to switch between them:

| `DATA_FOLDER` | Description |
|---|---|
| `'student_data'` | Photos taken by students in this class |
| `'kaggle_data'` | Kaggle finger-counting dataset (fallback) |

Both are organized the same way:

```
<DATA_FOLDER>/
    train/
        1/   2/   3/   4/   5/
    test/
        1/   2/   3/   4/   5/
```

This is a 5-class classification problem. Input: a grayscale 128x128 image. Output: one of five classes (1-5 fingers).

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Use GPU if available, otherwise CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

torch.manual_seed(1848)
print('Using device:', device)

## Loading the Data

Since our images are organized into one folder per class, we can use PyTorch's built-in `ImageFolder` dataset. No custom Dataset class needed. It automatically assigns labels based on folder names.

We apply three transforms to every image:
- `Grayscale()`: convert to single-channel grayscale
- `Resize((128, 128))`: make sure every image is the same size
- `ToTensor()`: convert the PIL image to a PyTorch tensor and scale pixel values from [0, 255] to [0.0, 1.0]

Note: `ImageFolder` assigns class indices alphabetically, so folder `"1"` gets label 0, `"2"` gets label 1, ..., `"5"` gets label 4.

In [ ]:
DATA_FOLDER = 'student_data'   # switch to 'kaggle_data' to use the Kaggle dataset

train_path = DATA_FOLDER + '/train'
test_path  = DATA_FOLDER + '/test'

transform = transforms.Compose([
    transforms.Grayscale(),        # 3-channel -> 1-channel
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(train_path, transform=transform)
test_dataset  = datasets.ImageFolder(test_path,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print('Dataset:  ', DATA_FOLDER)
print('Classes:  ', train_dataset.classes)   # should be ['1', '2', '3', '4', '5']
print('Training images:', len(train_dataset))
print('Test images:    ', len(test_dataset))

Let's look at a few images from the training set to make sure things loaded correctly.

In [ ]:
import random

samples_per_class = 5
n_classes         = 5

fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(10, 10))

for class_idx in range(n_classes):
    # ImageFolder stores all labels in .targets, so we can find indices without
    # loading every image
    class_indices = [i for i, label in enumerate(train_dataset.targets)
                     if label == class_idx]
    chosen = random.sample(class_indices, samples_per_class)

    for col, idx in enumerate(chosen):
        img, label = train_dataset[idx]
        axes[class_idx, col].imshow(img.squeeze(), cmap='gray')
        axes[class_idx, col].axis('off')
        if col == 0:
            axes[class_idx, col].set_ylabel(str(class_idx + 1) + ' fingers',
                                            fontsize=11)

plt.suptitle('Sample training images — 5 per class', fontsize=13)
plt.tight_layout()
plt.show()

## Defining the CNN

Our network follows the **Conv -> ReLU -> MaxPool** pattern, repeated one or more times, then fully-connected layers on top.

---

### `nn.Conv2d(in_channels, out_channels, kernel_size, padding=0)`

- **`in_channels`**: how many channels the input has. For the first layer this is 1 (grayscale) or 3 (color). For later conv layers it must match the `out_channels` of the previous conv layer.
- **`out_channels`**: how many filters to learn. Each filter produces one output feature map. More filters means more patterns the layer can detect, but more parameters.
- **`kernel_size`**: the size of each filter. `kernel_size=3` means each filter is a 3x3 grid of weights.
- **`padding`**: zeros added around the border before convolving. With `padding=1` and `kernel_size=3`, the output is the same spatial size as the input, so only the pooling layer shrinks the dimensions.

### `nn.MaxPool2d(kernel_size)`

- **`kernel_size`**: `MaxPool2d(2)` takes the max of each non-overlapping 2x2 tile, halving each spatial dimension.

---

With `padding=1` and `kernel_size=3`, the pattern is simple: each block doubles the channels and halves the spatial dimensions.

| Blocks | After conv+pool | `Linear` input size |
|---|---|---|
| 1 | 32 x 64 x 64 | `32 * 64 * 64` |


When adding a block, update the `Linear` input size to match.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(8 * 64 * 64, 128),
    nn.ReLU(),
    nn.Linear(128, 5)
)

model = model.to(device)
print(model)

## Training

The training loop is the same pattern we used for classification before:

1. Forward pass: compute predictions
2. Compute loss with `CrossEntropyLoss`
3. `optimizer.zero_grad()`: clear old gradients
4. `loss.backward()`: compute new gradients
5. `optimizer.step()`: update weights

One new thing: we call `.to(device)` on each batch to make sure the data is on the same device as the model.

In [ ]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 25

cnn_train_losses = []
cnn_test_losses  = []
cnn_train_accs   = []
cnn_test_accs    = []

for epoch in range(epochs):

    # ---- training pass ----
    model.train()
    total_loss    = 0
    train_correct = 0
    train_total   = 0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        y_hat = model(imgs)
        loss  = loss_fn(y_hat, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        train_correct += (y_hat.argmax(dim=1) == labels).sum().item()
        train_total   += imgs.size(0)

    # ---- test pass ----
    model.eval()
    test_loss    = 0
    test_correct = 0
    test_total   = 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            y_hat  = model(imgs)
            test_loss    += loss_fn(y_hat, labels).item()
            test_correct += (y_hat.argmax(dim=1) == labels).sum().item()
            test_total   += imgs.size(0)

    cnn_train_losses.append(total_loss    / len(train_loader))
    cnn_test_losses.append( test_loss     / len(test_loader))
    cnn_train_accs.append(  train_correct / train_total)
    cnn_test_accs.append(   test_correct  / test_total)

    print('Epoch', epoch + 1,
          ' | Train loss:', round(cnn_train_losses[-1], 4),
          ' Test loss:',    round(cnn_test_losses[-1],  4),
          ' | Train acc:',  round(cnn_train_accs[-1],   4),
          ' Test acc:',     round(cnn_test_accs[-1],    4))

In [ ]:
epoch_range = range(1, epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epoch_range, cnn_train_losses, label='Train')
ax1.plot(epoch_range, cnn_test_losses,  label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CNN Loss')
ax1.legend()

ax2.plot(epoch_range, cnn_train_accs, label='Train')
ax2.plot(epoch_range, cnn_test_accs,  label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('CNN Accuracy')
ax2.legend()

plt.tight_layout()
plt.show()

## Comparison: MLP vs CNN

How much does convolution actually help? Let's train a plain MLP (no convolutions, just flattened pixels into fully-connected layers) and compare accuracy, a confusion matrix, and per-class precision and recall.

A 128x128 grayscale image has 16,384 inputs. An MLP connects every pixel to every neuron with no notion of which pixels are neighbors. A CNN uses filters that slide across the image and share weights, which is a much better fit for image data.

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

mlp = nn.Sequential(
    nn.Flatten(),                       # 1 x 128 x 128  ->  16,384

    # --- Layer 1 ---
    nn.Linear(128 * 128, 256),
    nn.ReLU(),

    # --- Layer 2 (uncomment to add) ---
    # nn.Linear(256, 128),
    # nn.ReLU(),

    nn.Linear(256, 5)                   # <-- update input size if layers above change
).to(device)

print('CNN parameters:', count_params(model))
print('MLP parameters:', count_params(mlp))

In [ ]:
mlp_optimizer = optim.Adam(mlp.parameters())

mlp_train_losses = []
mlp_test_losses  = []
mlp_train_accs   = []
mlp_test_accs    = []

for epoch in range(epochs):

    # ---- training pass ----
    mlp.train()
    total_loss    = 0
    train_correct = 0
    train_total   = 0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        y_hat = mlp(imgs)
        loss  = loss_fn(y_hat, labels)

        mlp_optimizer.zero_grad()
        loss.backward()
        mlp_optimizer.step()

        total_loss    += loss.item()
        train_correct += (y_hat.argmax(dim=1) == labels).sum().item()
        train_total   += imgs.size(0)

    # ---- test pass ----
    mlp.eval()
    test_loss    = 0
    test_correct = 0
    test_total   = 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            y_hat  = mlp(imgs)
            test_loss    += loss_fn(y_hat, labels).item()
            test_correct += (y_hat.argmax(dim=1) == labels).sum().item()
            test_total   += imgs.size(0)

    mlp_train_losses.append(total_loss    / len(train_loader))
    mlp_test_losses.append( test_loss     / len(test_loader))
    mlp_train_accs.append(  train_correct / train_total)
    mlp_test_accs.append(   test_correct  / test_total)

    print('Epoch', epoch + 1,
          ' | Train loss:', round(mlp_train_losses[-1], 4),
          ' Test loss:',    round(mlp_test_losses[-1],  4),
          ' | Train acc:',  round(mlp_train_accs[-1],   4),
          ' Test acc:',     round(mlp_test_accs[-1],    4))

# ---- side-by-side comparison plots ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epoch_range, cnn_train_losses, label='CNN train')
ax1.plot(epoch_range, cnn_test_losses,  label='CNN test')
ax1.plot(epoch_range, mlp_train_losses, label='MLP train', linestyle='--')
ax1.plot(epoch_range, mlp_test_losses,  label='MLP test',  linestyle='--')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss: CNN vs MLP')
ax1.legend()

ax2.plot(epoch_range, cnn_train_accs, label='CNN train')
ax2.plot(epoch_range, cnn_test_accs,  label='CNN test')
ax2.plot(epoch_range, mlp_train_accs, label='MLP train', linestyle='--')
ax2.plot(epoch_range, mlp_test_accs,  label='MLP test',  linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy: CNN vs MLP')
ax2.legend()

plt.tight_layout()
plt.show()

### Accuracy, Confusion Matrix, and Precision/Recall

A single accuracy number hides a lot. The confusion matrix shows which classes get mixed up, for example, does the model confuse 3 fingers with 4 more than 1 with 5?

**Precision** for a class: of all the times the model predicted that class, how often was it right?

**Recall** for a class: of all the true examples of that class, how many did the model find?

Low precision means lots of false positives. Low recall means lots of false negatives.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def get_predictions(m, loader):
    m.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs  = imgs.to(device)
            preds = m(imgs).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

true_labels, cnn_preds = get_predictions(model, test_loader)
_,           mlp_preds = get_predictions(mlp,   test_loader)

print('Final CNN  train acc:', round(cnn_train_accs[-1], 4), ' test acc:', round(cnn_test_accs[-1], 4))
print('Final MLP  train acc:', round(mlp_train_accs[-1], 4), ' test acc:', round(mlp_test_accs[-1], 4))

In [ ]:
def plot_confusion_matrix(true, preds, title):
    cm      = confusion_matrix(true, preds)
    classes = ['1', '2', '3', '4', '5']

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted fingers')
    ax.set_ylabel('True fingers')
    ax.set_title(title)

    # Write counts inside each cell
    for i in range(5):
        for j in range(5):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)

    plt.tight_layout()
    plt.show()

plot_confusion_matrix(true_labels, cnn_preds, 'CNN Confusion Matrix')
plot_confusion_matrix(true_labels, mlp_preds, 'MLP Confusion Matrix')

In [ ]:
finger_names = ['1 finger', '2 fingers', '3 fingers', '4 fingers', '5 fingers']

print('CNN')
print(classification_report(true_labels, cnn_preds, target_names=finger_names))

print('MLP')
print(classification_report(true_labels, mlp_preds, target_names=finger_names))

## Visualizing Predictions

Some test images with the model's predictions. Green title = correct, red = wrong.

In [ ]:
model.eval()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.ravel()):
    img, true_label = test_dataset[i]

    with torch.no_grad():
        y_hat     = model(img.unsqueeze(0).to(device))  # add batch dimension
        pred_label = y_hat.argmax(dim=1).item()

    # labels are 0-indexed; add 1 to display finger count
    true_fingers = true_label + 1
    pred_fingers = pred_label + 1

    color = 'green' if pred_label == true_label else 'red'
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title('True: ' + str(true_fingers) + '  Pred: ' + str(pred_fingers), color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()